# DAT09 · 미션1 보고 형식과 업무시간 확인하기 (교수자·해설용 워크북 — 예시 풀이 포함)

- **예상 실습 시간**: 1.5시간 (전체 10개 미션 총 27시간 과정)
- **작업 폴더**: `DAT09_reference`
- **연결 파일**: 첫 셀에서 `DAT09.md`를 연결하세요.

> 학습자 배포 시에는 `TODO(n, step)` 상태인 `DAT09_m01_student.ipynb`만 배포하고, 본 워크북은 해설·채점 참고용으로 분리 보관하세요.


In [ ]:
from pathlib import Path, PurePosixPath
import sys, io, json, re, zipfile, base64, hashlib, importlib
from IPython.display import display, HTML
import html
import pandas as pd
ROLE = 'solution'
EXPECTED_SHA256 = 'a0adfd7b9a2416cd74af2fc99bb7978ad662eec8d9e40ce5b8a539977c2cb17e'
c = None

def connect_md(path):
    global c, ROOT, HERE
    path = Path(path)
    if path.name not in ('DAT09.md', 'DAT09.md'):
        raise ValueError('원자료 DAT09.md 또는 DAT09.md를 선택하세요. 정답코드 MD는 연결 파일이 아닙니다.')
    text = path.read_text(encoding='utf-8-sig')
    found = re.search(r'<!-- DAT09_RUNTIME_BUNDLE_B64:([A-Za-z0-9+/=]+) -->', text)
    if not found:
        raise ValueError('이 노트북용 원자료 블록이 없습니다. 함께 받은 DAT09.md를 사용하세요.')
    raw = base64.b64decode(found.group(1), validate=True)
    if hashlib.sha256(raw).hexdigest() != EXPECTED_SHA256:
        raise ValueError('자료와 노트북 버전이 다릅니다. 같은 묶음의 파일을 사용하세요.')
    ROOT = Path.cwd() / ('DAT09_reference' if ROLE == 'solution' else 'DAT09_practice')
    ROOT.mkdir(exist_ok=True)
    with zipfile.ZipFile(io.BytesIO(raw)) as archive:
        for info in archive.infolist():
            name = PurePosixPath(info.filename)
            if name.is_absolute() or '..' in name.parts:
                raise ValueError('허용되지 않은 경로입니다.')
            target = ROOT.joinpath(*name.parts)
            target.parent.mkdir(parents=True, exist_ok=True)
            target.write_bytes(archive.read(info))
    sys.path.insert(0, str(ROOT))
    for name in list(sys.modules):
        if name in ['dat09_common', 'dat09_ui'] or name.startswith('src.m'):
            del sys.modules[name]
    import dat09_common
    globals().update({name: getattr(dat09_common, name) for name in dir(dat09_common) if not name.startswith('_')})
    import collections, itertools, queue, threading, tempfile, importlib.util, shutil
    globals().update({k:v for k,v in locals().items() if k in ['collections','itertools','re','queue','threading','tempfile','importlib','shutil']})
    from dat09_ui import bars, heatmap, line_chart, dashboard_fragment
    globals().update(bars=bars,heatmap=heatmap,line_chart=line_chart,dashboard_fragment=dashboard_fragment)
    c = dat09_common.context(ROOT)
    HERE = ROOT.parent / 'instructor'
    for candidate in [Path.cwd()/'instructor', Path.cwd(), ROOT.parent/'instructor']:
        if (candidate/'aar_key.json').is_file():
            HERE = candidate
            c['instructor_root'] = candidate
            break
    print('자료 연결 완료:', ROOT)
    print('10개 미션·50개 함수·27시간 과정입니다. 미션1부터 실행하세요.')
    print('TODO가 남아 있으면 해당 단계에서 멈춥니다. src에는 작성한 함수를 저장합니다.')
    return c

DATA_MD = next((Path(p) for p in ('DAT09.md', 'DAT09.md') if Path(p).is_file()), None)
if DATA_MD is not None:
    connect_md(DATA_MD)
else:
    print('DAT09.md(또는 DAT09.md)를 이 노트북과 같은 폴더에 두고 첫 셀을 다시 실행하세요.')
    print("다른 경로에 있다면 connect_md('/자료가_있는_경로/DAT09.md')를 실행하세요.")


## 미션1 보고 형식과 업무시간 확인하기

예상 1.5시간입니다.

#### 이번 미션의 목표

세 파일이 어떤 재고를 보고하는지 확인하고, 뒤의 코드 작성에서 지킬 변환 규칙과 개선 순서를 정합니다.

### 1단계 세 재고 보고의 항목과 뜻 비교하기

#### 이 단계에서 하는 일

보급·정비·탄약이라는 파일 이름만 보고 수량을 합치면 같은 재고를 세 번 더할 수 있습니다. 이번 자료에서는 세 출처가 같은 거점과 품목의 보유 수량을 각각 보고합니다. 먼저 같은 대상을 비교할 수 있는지 확인해야 합니다.

#### 시작 전 확인

노트북 첫 셀로 자료를 연결한 상태에서 시작합니다. 노트북의 m01_s1(c)를 작성하세요. c는 같은 작업의 자료와 함수를 담는 사전입니다. 필요한 결과를 c에 보관하고 화면에 보여 줄 표를 반환합니다.

#### 사용할 자료

data/supply.csv, data/maintenance.json, data/ammunition.tsv와 config/source_contracts.json을 사용합니다. source_contracts의 field_mapping은 출처별 항목 이름을 공통 이름으로 바꾸는 표이며, refresh_minutes는 정해 놓은 갱신 간격입니다.

#### 작업 순서

- CSV는 쉼표, TSV는 탭으로 항목을 나누어 읽고, JSON은 records 안의 보고 목록을 읽으세요. 파일 이름만 목록에 올리지 말고 실제 보고 행을 읽습니다.

- 파일별로 출처, 형식, 행 수, 항목 이름, 첫 보고의 코드·수량·단위·날짜 표기, 갱신 간격을 한 표에 모으세요. SHA256 값도 남깁니다. SHA256은 나중에 같은 파일을 사용했는지 비교하는 식별값입니다.

- 동일한 거점·품목·기준시각의 보고를 세 출처에서 하나씩 찾아 비교하세요. 보고된 수량의 뜻은 on_hand, 즉 그 시점의 보유 수량입니다. 같은 재고의 재보고를 합산하지 않는 이유를 기록하세요.


#### 저장할 결과

source_spec.csv와 세 파일의 표본을 남깁니다. 표본은 다음 단계에서 다시 읽거나 c['sample_sources']에 보관해 사용할 수 있습니다.

#### 직접 확인하는 방법

파일마다 720행인지, 전체가 2,160행인지 직접 세세요. 예를 들어 동일 재고가 세 파일에서 각각 80EA로 보고돼도 재고는 240EA가 되지 않습니다.

#### 실행과 기록

이 미션의 다섯 함수를 작성한 뒤 코드셀을 실행합니다. 반환 표는 outputs/M01/step_01.csv에 저장됩니다. 전체 결과는 같은 폴더의 DAT09_M01_result.html에서 확인합니다. 사람이 확인한 내용은 review_notes.md에 단계 번호·사례·근거·판단을 직접 적으세요. 앞 단계 자료가 없으면 이전 미션부터 순서대로 실행합니다.

### 2단계 출처별 코드를 같은 품목과 거점으로 연결하기

#### 이 단계에서 하는 일

같은 품목도 출처에 따라 코드가 다릅니다. 공통 코드를 붙이되 원래 코드를 잃지 않아야 잘못 연결한 행을 되짚을 수 있습니다. 또한 무엇을 하나의 보고로 볼지 정해야 중복을 판별할 수 있습니다.

#### 시작 전 확인

미션1의 1단계까지 작성·확인한 결과를 사용합니다. 노트북의 m01_s2(c)를 작성하세요. c는 같은 작업의 자료와 함수를 담는 사전입니다. 필요한 결과를 c에 보관하고 화면에 보여 줄 표를 반환합니다.

#### 사용할 자료

data/catalog.json의 mapping, items, sites와 1단계의 비교표를 사용합니다. mapping은 각 출처의 원래 품목·거점 코드를 공통 코드로 연결합니다.

#### 작업 순서

- 출처별 원래 품목 코드와 거점 코드를 공통 item_id, site_id로 바꾼 표를 만드세요. 출처마다 품목 20개와 거점 6개의 조합 120개, 전체 360개 조합을 검사합니다.

- 공통 코드에서 출처별 원래 코드로 다시 돌아오는 표도 만드세요. 서로 다른 원래 품목이 잘못 하나로 합쳐지거나, 돌아온 코드가 달라진 조합을 따로 셉니다.

- 재고 이력을 구별하는 조합은 site_id·item_id·metric·as_of로 정하세요. 출처의 관측 보고를 구별할 때는 source를 더합니다. 실시간 현재값은 source·site_id·item_id·metric별로 유지하고 버전을 비교합니다. 이 세 구분을 key_contract.json에 적으세요.


#### 저장할 결과

360개 변환·복원 결과를 단계 반환 표로 남기고, 식별 기준과 충돌 건수를 key_contract.json에 저장합니다.

#### 직접 확인하는 방법

360개 조합에서 변환한 뒤 복원한 코드가 모두 같아야 합니다. 서로 다른 기준시각의 보고가 하나의 이력으로 덮어써지지 않는지도 확인하세요.

#### 실행과 기록

이 미션의 다섯 함수를 작성한 뒤 코드셀을 실행합니다. 반환 표는 outputs/M01/step_02.csv에 저장됩니다. 전체 결과는 같은 폴더의 DAT09_M01_result.html에서 확인합니다. 사람이 확인한 내용은 review_notes.md에 단계 번호·사례·근거·판단을 직접 적으세요. 앞 단계 자료가 없으면 이전 미션부터 순서대로 실행합니다.

### 3단계 업무시간을 계산하고 지연이 큰 단계 찾기

#### 이 단계에서 하는 일

자동화 효과를 말하려면 현재 어디에서 시간이 드는지부터 알아야 합니다. 사람이 작업하는 시간과 다음 처리를 기다리는 시간은 개선 방법이 다릅니다. 이번 파일의 시간은 실제 부대 실측값이 아닌 실습용 가정입니다.

#### 시작 전 확인

미션1의 2단계까지 작성·확인한 결과를 사용합니다. 노트북의 m01_s3(c)를 작성하세요. c는 같은 작업의 자료와 함수를 담는 사전입니다. 필요한 결과를 c에 보관하고 화면에 보여 줄 표를 반환합니다.

#### 사용할 자료

data/workflow.csv의 12개 업무 단계와 processing_min, waiting_min을 사용합니다. 두 열의 단위는 분입니다.

#### 작업 순서

- 각 단계의 처리시간과 대기시간을 숫자로 읽고, 단계별 합계와 전체 합계를 계산하세요. 전체 처리시간과 전체 대기시간도 구분해 적습니다.

- 처리시간이 30분 이상이거나 대기시간이 30분 이상이면 해당 단계를 지연이 큰 단계로 표시하세요. 두 조건에 모두 해당해도 단계 수는 한 번만 셉니다.

- 단계별 합계 시간을 막대그래프로 그리세요. 개선 후 전체 시간을 50분으로 가정하고 (현재시간-개선후시간)/현재시간으로 절감률을 구합니다. 표와 그래프에 실습용 가정임을 표시하세요.


#### 저장할 결과

workflow_comparison.csv에 12단계의 두 시간과 합계·판정값을 저장하고, workflow_assumptions.json에 전체 합계·개선 가정·절감률을 저장합니다.

#### 직접 확인하는 방법

처리 210분, 대기 340분, 합계 550분과 지연이 큰 단계 8개를 확인하세요. 550분에서 50분으로 줄인다는 가정의 절감률은 약 90.91%입니다.

#### 실행과 기록

이 미션의 다섯 함수를 작성한 뒤 코드셀을 실행합니다. 반환 표는 outputs/M01/step_03.csv에 저장됩니다. 전체 결과는 같은 폴더의 DAT09_M01_result.html에서 확인합니다. 사람이 확인한 내용은 review_notes.md에 단계 번호·사례·근거·판단을 직접 적으세요. 앞 단계 자료가 없으면 이전 미션부터 순서대로 실행합니다.

### 4단계 현재 환경을 평가하고 먼저 고칠 일 정하기

#### 이 단계에서 하는 일

뒤의 미션을 모두 끝낸 것처럼 점수를 주면 무엇이 개선됐는지 알 수 없습니다. 지금까지 확보한 근거만으로 현재 상태를 평가하고, 다음 작업의 순서를 정합니다.

#### 시작 전 확인

미션1의 3단계까지 작성·확인한 결과를 사용합니다. 노트북의 m01_s4(c)를 작성하세요. c는 같은 작업의 자료와 함수를 담는 사전입니다. 필요한 결과를 c에 보관하고 화면에 보여 줄 표를 반환합니다.

#### 사용할 자료

1~3단계의 비교표·코드 검사·업무시간 결과와 config/actor_registry.json을 사용합니다. 평가 영역은 파일 연결, 코드 표준, 자료 품질, 갱신 적시성, 권한과 책임, 다시 실행하기의 여섯 가지입니다.

#### 작업 순서

- 각 영역에 현재 확인한 사실, 부족한 점, 담당자, 다음 작업을 적으세요. 근거가 없으면 0점, 문서나 수동 확인만 있으면 1점, 별도 시험으로 정상·오류 상황까지 확인했으면 2점을 줍니다.

- 점수 옆에 확인한 파일·행 또는 시험 이름을 적으세요. 앞으로 만들 계획만으로 점수를 올리지 않습니다. 코드를 한 번 실행했다는 사실만으로 오류 대응까지 검증한 것으로 보지 않습니다.

- 가장 먼저 고칠 작업 세 개를 고르고 순서, 담당자, 선행 작업, 완료를 확인할 조건을 적으세요. 예를 들어 코드 변환을 먼저 끝내야 품질 집계를 할 수 있다는 관계를 드러냅니다.


#### 저장할 결과

environment_baseline.csv에 여섯 영역의 평가를, improvement_backlog.json에 우선 작업 세 개를 저장합니다.

#### 직접 확인하는 방법

여섯 영역이 빠짐없이 있고 점수는 0~2 범위인지 확인하세요. 세 작업의 완료 조건은 “개선한다”가 아니라 어떤 파일이나 시험이 나오면 끝나는지로 적습니다.

#### 실행과 기록

이 미션의 다섯 함수를 작성한 뒤 코드셀을 실행합니다. 반환 표는 outputs/M01/step_04.csv에 저장됩니다. 전체 결과는 같은 폴더의 DAT09_M01_result.html에서 확인합니다. 사람이 확인한 내용은 review_notes.md에 단계 번호·사례·근거·판단을 직접 적으세요. 앞 단계 자료가 없으면 이전 미션부터 순서대로 실행합니다.

### 5단계 다음 담당자가 사용할 변환 규칙 작성하기

#### 이 단계에서 하는 일

이 문서는 다음 미션에서 파일을 읽고 값을 바꾸는 기준입니다. “표준화한다”는 말만으로는 어떤 값을 어디까지 바꿔도 되는지 알 수 없습니다. 다른 사람이 이 문서만 보고 같은 처리를 할 수 있게 씁니다.

#### 시작 전 확인

미션1의 4단계까지 작성·확인한 결과를 사용합니다. 노트북의 m01_s5(c)를 작성하세요. c는 같은 작업의 자료와 함수를 담는 사전입니다. 필요한 결과를 c에 보관하고 화면에 보여 줄 표를 반환합니다.

#### 사용할 자료

세 파일의 비교표, catalog.json, source_contracts.json과 앞 단계의 식별 기준을 사용합니다. 새 규칙을 임의로 추가하지 말고 제공 자료에서 확인한 기준을 옮깁니다.

#### 작업 순서

- spec.md에 입력 파일과 공통 항목의 대응, 품목·거점 코드표, 재고 이력과 출처별 관측의 식별 기준을 적으세요.

- 기본 단위 EA, 등록된 BOX의 10EA 환산, 시간대가 있는 시각의 UTC 변환, 60분 초과 지연 표시 방법을 예시와 함께 쓰세요. 원래 코드·값·단위·시각은 보존합니다.

- 결측 수량은 0으로 만들지 않고, 미등록 코드·단위·잘못된 날짜는 확인할 때까지 보류 또는 격리한다고 적으세요. 다른 두 출처가 같아도 보완 후보일 뿐 승인값이 아님을 명시합니다.

- 동료에게 stock_amount라는 새 항목과 출고수량이라는 새 항목을 각각 제시하세요. 전자는 재고와 같은 뜻인지 확인한 뒤 등록하고, 후자는 재고량으로 바로 연결하지 않는 이유를 검토 기록에 남깁니다.


#### 저장할 결과

spec.md와 검토자·검토일·질문·답변·수정 사항을 적은 검토 기록을 저장합니다. 혼자 진행하면 교관이 동료 검토를 대신할 수 있습니다.

#### 직접 확인하는 방법

파일·항목·코드·단위·시각·결측·지연·후보·승인의 기준이 모두 있는지 체크하세요. “적절히 처리한다”처럼 선택을 남기는 문장은 구체적인 조건으로 바꿉니다.

#### 실행과 기록

이 미션의 다섯 함수를 작성한 뒤 코드셀을 실행합니다. 반환 표는 outputs/M01/step_05.csv에 저장됩니다. 전체 결과는 같은 폴더의 DAT09_M01_result.html에서 확인합니다. 사람이 확인한 내용은 review_notes.md에 단계 번호·사례·근거·판단을 직접 적으세요. 앞 단계 자료가 없으면 이전 미션부터 순서대로 실행합니다.



In [ ]:
if c is None:
    raise RuntimeError("첫 셀에서 DAT09.md(또는 DAT09.md)를 먼저 연결하세요.")

# STEP 01 세 재고 보고의 항목과 뜻 비교하기
def m01_s1(c):
    """세 재고 보고의 항목과 뜻 비교하기."""
    root = c['root']
    out = folder(c, 1)
    files = {
        'A': ('csv', root / 'data/supply.csv', csv_rows(root / 'data/supply.csv')),
        'B': ('json', root / 'data/maintenance.json', read_json(root / 'data/maintenance.json')['records']),
        'C': ('tsv', root / 'data/ammunition.tsv', csv_rows(root / 'data/ammunition.tsv', delimiter='\t')),
    }
    spec_rows = []
    for src, (fmt, path, rows) in files.items():
        first = rows[0]
        fm = c['contract'][src]['field_mapping']
        inv_fm = {v: k for k, v in fm.items()}
        spec_rows.append({
            'source': src,
            'format': fmt,
            'file': str(path.relative_to(root)),
            'row_count': len(rows),
            'sha256': sha(path),
            'columns': ', '.join(first.keys()),
            'metric_meaning': 'on_hand(시점 보유 수량 · 합산 금지)',
            'first_row_id': first['row_id'],
            'first_item_code': first[inv_fm['item_code']],
            'first_site_code': first[inv_fm['site_code']],
            'first_qty': first[inv_fm['quantity']],
            'first_unit': first['unit'],
            'first_as_of': first[inv_fm['as_of']],
            'refresh_minutes': c['contract'][src]['refresh_minutes'],
        })
    csv_out(out / 'source_spec.csv', spec_rows)

    # 동일 거점(U006)·품목(I020)·기준시각(2026-10-01 14:00+09:00) 표본 추출
    c['sample_sources'] = {
        'A': files['A'][2][-1],
        'B': files['B'][2][-1],
        'C': files['C'][2][-1],
    }
    notes = [
        '# M01 검토 기록 (review_notes.md)',
        '',
        '## 1단계: 세 출처 항목 비교 및 중복 합산 금지 이유',
        '- 확인 파일 및 행 수: data/supply.csv(720행), data/maintenance.json records(720행), data/ammunition.tsv(720행), 총 2,160행.',
        '- 동일 대상 표본: A-0719(S020/DEP006, 90 EA), B-0719(PART_020/UNIT-06, 9 BOX=90 EA), C-0719(T-0020/X006, 90 EA).',
        '- 합산하지 않는 이유: 세 파일은 서로 다른 창고가 아니라 같은 (거점, 품목, 기준시각)의 on_hand 보유 수량을 재보고하므로, 세 수량을 더하면 재고가 3배로 과대 계상된다.',
    ]
    (out / 'review_notes.md').write_text('\n'.join(notes) + '\n', encoding='utf-8')
    return spec_rows

# STEP 02 출처별 코드를 같은 품목과 거점으로 연결하기
def m01_s2(c):
    """출처별 코드를 같은 품목과 거점으로 연결하기."""
    out = folder(c, 1)
    cat = c['catalog']
    rows = []
    collisions = 0
    for src in ['A', 'B', 'C']:
        item_map = cat['mapping'][src]['item']
        site_map = cat['mapping'][src]['site']
        rev_item = {v: k for k, v in item_map.items()}
        rev_site = {v: k for k, v in site_map.items()}
        if len(rev_item) != len(item_map) or len(rev_site) != len(site_map):
            collisions += 1
        for raw_item, item_id in item_map.items():
            for raw_site, site_id in site_map.items():
                restored_item = rev_item[item_id]
                restored_site = rev_site[site_id]
                ok = (restored_item == raw_item) and (restored_site == raw_site)
                if not ok:
                    collisions += 1
                rows.append({
                    'source': src,
                    'raw_item': raw_item,
                    'raw_site': raw_site,
                    'item_id': item_id,
                    'site_id': site_id,
                    'restored_item': restored_item,
                    'restored_site': restored_site,
                    'roundtrip_ok': ok,
                })
    contract_doc = {
        'logical_history_key': ['site_id', 'item_id', 'metric', 'as_of'],
        'source_observation_key': ['source', 'site_id', 'item_id', 'metric', 'as_of'],
        'current_state_key': ['source', 'site_id', 'item_id', 'metric'],
        'version_comparison': 'higher version wins; same version different payload -> CONFLICT',
        'roundtrip_checked': len(rows),
        'roundtrip_all_ok': all(r['roundtrip_ok'] for r in rows),
        'collisions': collisions,
    }
    write_json(out / 'key_contract.json', contract_doc)
    return rows

# STEP 03 업무시간을 계산하고 지연이 큰 단계 찾기
def m01_s3(c):
    """업무시간을 계산하고 지연이 큰 단계 찾기."""
    out = folder(c, 1)
    raw = csv_rows(c['root'] / 'data/workflow.csv')
    rows = []
    total_proc = 0
    total_wait = 0
    bottleneck_count = 0
    for r in raw:
        p = int(r['processing_min'])
        w = int(r['waiting_min'])
        s = p + w
        bn = (p >= 30) or (w >= 30)
        total_proc += p
        total_wait += w
        if bn:
            bottleneck_count += 1
        rows.append({
            'step': int(r['step']),
            'activity': r['activity'],
            'processing_min': p,
            'waiting_min': w,
            'total_min': s,
            'is_bottleneck_ge30': bn,
            'basis': '실습용 가정(부대 실측값 아님)',
        })
    current_total = total_proc + total_wait
    assumed_after = 50
    reduction_pct = round((current_total - assumed_after) / current_total * 100, 2)
    assumptions = {
        'basis': '실습용 가정(부대 실측값 아님)',
        'step_count': len(rows),
        'total_processing_min': total_proc,
        'total_waiting_min': total_wait,
        'current_total_min': current_total,
        'bottleneck_steps_count': bottleneck_count,
        'assumed_after_min': assumed_after,
        'assumed_reduction_pct': reduction_pct,
    }
    csv_out(out / 'workflow_comparison.csv', rows)
    write_json(out / 'workflow_assumptions.json', assumptions)
    c.setdefault('stage_html', {})[(1, 3)] = bars(
        [f"S{r['step']}" for r in rows],
        [r['total_min'] for r in rows],
        f"단계별 소요시간(실습용 가정 · 합계 {current_total}분 → {assumed_after}분 가정, 절감률 {reduction_pct}%)",
        unit='분(가정)',
        colors=['#c85a32' if r['is_bottleneck_ge30'] else '#3c719b' for r in rows],
    )
    c['workflow'] = assumptions
    return rows

# STEP 04 현재 환경을 평가하고 먼저 고칠 일 정하기
def m01_s4(c):
    """현재 환경을 평가하고 먼저 고칠 일 정하기."""
    out = folder(c, 1)
    actors = read_json(c['root'] / 'config/actor_registry.json')
    baseline = [
        {
            'area': '파일 연결',
            'score': 1,
            'current_fact': '3종 파일(720행씩 총 2,160행) 구조 및 SHA256 비교 완료',
            'gap': '자동 적재·스키마 변경 방어 시험 미구현',
            'owner': 'ANALYST_1',
            'evidence': 'outputs/M01/source_spec.csv',
            'next_action': '표준 읽기 함수(load_fn) 및 변경 시험 구현',
        },
        {
            'area': '코드 표준',
            'score': 1,
            'current_fact': '360조합 코드 왕복 매핑 및 key_contract.json 정의 완료',
            'gap': '단위 환산(BOX->10EA)·UTC 변환·격리/보류 처리 미구현',
            'owner': 'ANALYST_1',
            'evidence': 'outputs/M01/key_contract.json, step_02.csv',
            'next_action': '정규화 함수(normalize_fn) 및 status_fn 구현',
        },
        {
            'area': '자료 품질',
            'score': 0,
            'current_fact': '결측·오류·지연 정량 지표 및 품질 산식 미구현',
            'gap': '2,160 슬롯 기준 C·V·T·K·Q 및 검증 규칙 미적용',
            'owner': 'ANALYST_2',
            'evidence': '미측정(M01 시점 근거 없음)',
            'next_action': '15규칙 검증 및 4대 품질 지수 산출 구현',
        },
        {
            'area': '갱신 적시성',
            'score': 0,
            'current_fact': '수동 550분 가정 확인(대기 340분), 실시간 이벤트 처리기 미구현',
            'gap': '중복·과거·충돌·60분 초과 지연 판별 및 순차 갱신 부재',
            'owner': 'SERVICE_1',
            'evidence': 'outputs/M01/workflow_assumptions.json(가정치)',
            'next_action': 'SyncEngine 및 6시점 갱신 검증 구현',
        },
        {
            'area': '권한과 책임',
            'score': 1,
            'current_fact': f"역할 레지스트리 {len(actors)}개 계정 확인",
            'gap': '승인 상태기계·고영향 2단계·감사 해시 체인 미구현',
            'owner': 'STEWARD_1',
            'evidence': 'config/actor_registry.json',
            'next_action': 'ApprovalWorkflow 및 감사 체인 검증 구현',
        },
        {
            'area': '다시 실행하기',
            'score': 1,
            'current_fact': '노트북 M01 수동 실행 가능',
            'gap': '새 프로세스 일괄 실행(run_dat09.py) 및 장애 복구 시험 부재',
            'owner': 'STAFF_1',
            'evidence': 'DAT09_m01_student.ipynb / M01 실행',
            'next_action': 'CLI 파이프라인 통합 및 UAT·인수 점검 완료',
        },
    ]
    backlog = [
        {
            'priority': 1,
            'task': '표준 수집·코드/단위/UTC 정규화·상태 원장 구축',
            'owner': 'ANALYST_1',
            'prerequisite': 'outputs/M01/spec.md 및 key_contract.json',
            'done_condition': 'outputs/M02/normalized.csv, status_ledger.csv 및 8종 이상 시험 통과',
        },
        {
            'priority': 2,
            'task': 'SQLite 3계층 저장 및 15종 검증 룰·중복 합산 방지 뷰 구현',
            'owner': 'ANALYST_2',
            'prerequisite': 'M02 normalized.csv 및 status_fn',
            'done_condition': 'outputs/M03/integration.db, validation_issues.csv 및 45개 rule_cases 일치',
        },
        {
            'priority': 3,
            'task': '이벤트 동기화(SyncEngine)·품질 지수·승인 워크플로우 구현',
            'owner': 'STEWARD_1',
            'prerequisite': 'M03 integration.db 및 logical_rows',
            'done_condition': 'outputs/M04~M07 산출물 및 기본 승인 0건·10종 공격 차단 확인',
        },
    ]
    csv_out(out / 'environment_baseline.csv', baseline)
    write_json(out / 'improvement_backlog.json', backlog)
    c['environment_baseline'] = baseline
    return baseline

# STEP 05 다음 담당자가 사용할 변환 규칙 작성하기
def m01_s5(c):
    """다음 담당자가 사용할 변환 규칙 작성하기."""
    out = folder(c, 1)
    spec_md = """# DAT09 표준 변환 및 처리 명세 (spec.md)

1. **입력 파일 및 공통 항목 대응**: supply.csv(A), maintenance.json(B), ammunition.tsv(C)는 `source_contracts.json`의 `field_mapping`으로 `item_code`, `site_code`, `quantity`, `as_of`에 대응한다.
2. **품목·거점 코드표**: `catalog.json`의 등록된 20개 품목(`I001`~`I020`)과 6개 거점(`U001`~`U006`)으로 매핑하되 `raw_item`, `raw_site`, `raw_qty`, `raw_unit`, `raw_as_of` 원값을 보존한다.
3. **식별 기준**: 재고 이력은 `(site_id, item_id, metric, as_of)`, 출처 관측은 `(source, site_id, item_id, metric, as_of)`, 현재 상태는 `(source, site_id, item_id, metric)` + 버전 비교로 구별한다. 같은 재고의 세 출처 수량은 합산하지 않는다.
4. **단위 환산**: 기본 단위는 `EA`이며 등록된 `BOX`만 `box_to_ea=10`을 곱해 `quantity_ea`로 환산한다. 미등록 단위는 임의 환산하지 않고 `HELD`로 보류한다.
5. **시각 정규화 및 지연**: `as_of`, `observed_at`, `received_at`은 출처별 시간대 포맷으로 읽어 UTC로 변환한다. 시간대가 없거나 해석 불가면 `QUARANTINED`로 격리한다. `as_of - observed_at > 60분`이면 `STALE`(지연)으로 표시한다.
6. **결측 처리**: 공란·`NULL`·`N/A`는 `0`으로 바꾸지 않으며 `HELD`로 둔다.
7. **오류 격리**: 미등록 품목/거점, 날짜 오류, 비숫자, 음수, 비정수 `EA`는 `QUARANTINED`로 격리한다.
8. **보완 후보**: 동일 이력 키의 나머지 두 출처가 모두 `ACCEPTED`, `0~60분` 이내 관측, 동일 `quantity_ea`일 때만 후보를 제안한다.
9. **승인 원칙**: 후보는 `approved=False` 미확정 상태이며 데이터관리관 모의 승인 전까지 `approved_master`에 확정하지 않는다.
"""
    (out / 'spec.md').write_text(spec_md, encoding='utf-8')

    review_path = out / 'review_notes.md'
    prev = review_path.read_text(encoding='utf-8') if review_path.exists() else ''
    review_add = """
## 5단계: 변환 명세 동료 검토 기록
- 검토자: ANALYST_2 (동료 검토 모의) / 검토일: 2026-10-01
- 질문 1 (`stock_amount` 신규 항목): `on_hand` 보유 수량과 같은 뜻인지 출처 계약 확인 후 `field_mapping`에 등록해야 허용 가능함.
- 질문 2 (`출고수량` 신규 항목): 시점 보유 재고(`on_hand`)가 아니라 기간 이동량(flow)이므로 재고량으로 바로 연결하면 총보유량이 왜곡되어 거부해야 함.
- 수정 사항: `spec.md` 1번·4번·6번에 원값 보존, 결측!=0, 미등록 단위/새 항목 임의 연결 금지 명시.
"""
    review_path.write_text(prev + review_add, encoding='utf-8')

    rows = [
        {'topic': '입력 파일·항목 매핑', 'rule': 'source_contracts.json field_mapping 적용 및 원값 보존', 'status': 'SPECIFIED'},
        {'topic': '품목·거점 코드', 'rule': 'catalog.json 등록 20품목·6거점만 연결, 미등록은 QUARANTINED', 'status': 'SPECIFIED'},
        {'topic': '식별 키 기준', 'rule': '이력(4키) · 관측(5키) · 현재상태(4키+버전) 구분, 합산 금지', 'status': 'SPECIFIED'},
        {'topic': '단위 환산', 'rule': 'EA 유지, 등록 BOX=10EA 환산, 미등록 단위는 HELD', 'status': 'SPECIFIED'},
        {'topic': '시각·지연 기준', 'rule': '명시적 시간대만 UTC 변환, >60분은 STALE 표시', 'status': 'SPECIFIED'},
        {'topic': '결측·오류 구분', 'rule': '결측(공란/NULL/N/A)!=0은 HELD, 음수/날짜오류 등은 QUARANTINED', 'status': 'SPECIFIED'},
        {'topic': '후보·승인 분리', 'rule': '두 정상/적시 출처 일치 시 후보 제안(approved=False), 승인 전 미확정', 'status': 'SPECIFIED'},
        {'topic': '동료 검토(신규 항목)', 'rule': 'stock_amount는 의미 확인 후 등록, 출고수량은 재고 매핑 거부', 'status': 'REVIEWED'},
    ]
    return rows

result_html = run_mission(c, 1, [m01_s1, m01_s2, m01_s3, m01_s4, m01_s5])
document = result_html.read_text(encoding="utf-8")
display(HTML('<iframe title="DAT09 단계별 결과" style="width:100%;height:1100px;border:1px solid #ccd5dd" sandbox="allow-scripts allow-downloads allow-same-origin" srcdoc="'+html.escape(document,quote=True)+'"></iframe>'))
